# Phase 1 — Dataset Curation & Exploration

This notebook builds the unified curated dataset by:
1. **Downloading** the Lakh MIDI Dataset (LMD-matched, LMD-aligned, MSD HDF5, match scores)
2. Linking MIDI files with MSD HDF5 metadata via `LakhMSDLinker`
3. Filtering by DTW match score (`min_score ≥ 0.70`)
4. Extracting harmonic features per MIDI using `MIDIHarmonicAnalyzer`
5. Saving the resulting DataFrame to Parquet for downstream phases

| Asset | Size | Destination |
|---|---|---|
| `lmd_matched.tar.gz` | ~1.5 GB | `data/raw/lmd_matched/` |
| `lmd_aligned.tar.gz` | ~1.6 GB | `data/raw/lmd_aligned/` |
| `lmd_matched_h5.tar.gz` | ~2.5 GB | `data/raw/lmd_matched_h5/` |
| `match_scores.json` (matched) | ~9 MB | `data/raw/match_scores.json` |
| `match_scores.json` (aligned) | ~9 MB | `data/raw/match_scores_aligned.json` |

> **Kernel:** select **SONATAM (.venv)** in the kernel picker (top-right).  
> Run cell 2 to confirm you are in the right environment, then cell 3 to download all data.


In [1]:
# ── Kernel / environment sanity check ─────────────────────────────────────
# This cell confirms you are running inside the correct .venv.
# If it prints a path outside this project, switch kernel to "SONATAM (.venv)".
import sys
from pathlib import Path
from sonata.notebook_utils import rp

# Build the interpreter path from sys.prefix (not sys.executable) to avoid
# following the symlink all the way to the system Python binary.
_prefix = Path(sys.prefix)                         # .../SONATAM/.venv
_interp = _prefix / "bin" / Path(sys.executable).name

print("Python :", rp(_interp))   # → .venv/bin/python
print("Prefix :", rp(_prefix))   # → .venv

try:
    import sonata
    _pkg = Path(sonata.__file__).parent
    print(f"sonata  : {sonata.__version__}  ✓  ({rp(_pkg)})")
except ImportError:
    print("sonata  : NOT FOUND — switch to the 'SONATAM (.venv)' kernel")


Python : .venv/bin/python
Prefix : .venv
sonata  : 0.1.0  ✓  (src/sonata)


## Step 1 — Download raw data

Run the cell below **once**. It calls `scripts/download_lmd.py` which:
- Streams each archive with a progress bar
- Skips files that already exist (`--skip-existing` is on by default)
- Unpacks each `.tar.gz` in-place into `data/raw/<name>/`
- Writes `data/raw/manifest.json` listing all paths

To download only specific assets add e.g. `--only match_scores match_scores_aligned`.


In [3]:
# ── Download LMD + MSD data ───────────────────────────────────────────────
# Safe to re-run: already-downloaded files are skipped automatically.
# Remove --skip-existing (or pass --no-skip) to force a fresh re-download.
# All paths printed below are relative to the project root.
import subprocess, sys
from sonata.notebook_utils import project_root

result = subprocess.run(
    [sys.executable, "scripts/download_lmd.py", "--skip-existing"],
    cwd=str(project_root),   # always run from the project root
    capture_output=False,
)
if result.returncode != 0:
    print("⚠  Download script exited with errors — check output above.")
else:
    print("✓  Download step complete.")



  SONATAM — Lakh MIDI Dataset downloader
  Destination : data/raw
  Assets      : lmd_matched, lmd_aligned, lmd_matched_h5, match_scores, match_scores_aligned


── lmd_matched  ──  LMD-matched  (~45 k MIDI files matched to MSD)
    ↓  http://hog.ee.columbia.edu/craffel/lmd/lmd_matched.tar.gz
       → data/raw/lmd_matched.tar.gz  (1.3 GB)
    ↓  http://hog.ee.columbia.edu/craffel/lmd/lmd_matched.tar.gz
       → data/raw/lmd_matched.tar.gz  (1.3 GB)


  6%|██▎                                   | 80.0M/1.31G [00:10<01:55, 11.5MB/s]Traceback (most recent call last):
  File "/home/pfanyka/Desktop/MASTERS/sem_2/KG_DL_PROJECT/SONATAM/scripts/download_lmd.py", line 290, in <module>
    main()
    ~~~~^^
  File "/home/pfanyka/Desktop/MASTERS/sem_2/KG_DL_PROJECT/SONATAM/scripts/download_lmd.py", line 257, in main
    _download(asset["url"], archive, skip_existing=args.skip_existing)
    ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/pfanyka/Desktop/MASTERS/sem_2/KG_DL_PROJECT/SONATAM/scripts/download_lmd.py", line 139, in _download
    block = resp.read(chunk)
  File "/usr/lib/python3.14/http/client.py", line 484, in read
    s = self.fp.read(amt)
  File "/usr/lib/python3.14/socket.py", line 725, in readinto
    return self._sock.recv_into(b)
           ~~~~~~~~~~~~~~~~~~~~^^^
KeyboardInterrupt
  6%|██▎                                   | 80.0M/1.31G [00:10<02:47, 7.88MB/s]
Traceback (most recent call last)

KeyboardInterrupt: 

## Step 2 — Load config & verify paths


In [3]:
# ── Imports & config ──────────────────────────────────────────────────────
import json
from pathlib import Path

import pandas as pd
import numpy  as np
import matplotlib.pyplot as plt

from sonata.config.settings  import CFG
from sonata.data             import MIDIHarmonicAnalyzer, LakhMSDLinker
from sonata.notebook_utils   import rp, show_paths  # relative-path display helpers

# Verify raw data paths exist
RAW = Path(CFG['data']['raw_dir'])
checks = {
    "lmd_matched":    RAW / "lmd_matched",
    "lmd_aligned":    RAW / "lmd_aligned",
    "lmd_matched_h5": RAW / "lmd_matched_h5",
    "match_scores":   RAW / "match_scores.json",
}
print("Config loaded — raw data status:")
for name, path in checks.items():
    status = "✓" if path.exists() else "✗  MISSING — run the download cell first"
    print(f"  {status}  {rp(path)}")


Config loaded — raw data status:
  ✗  MISSING — run the download cell first  notebooks/data/raw/lmd_matched
  ✗  MISSING — run the download cell first  notebooks/data/raw/lmd_aligned
  ✗  MISSING — run the download cell first  notebooks/data/raw/lmd_matched_h5
  ✗  MISSING — run the download cell first  notebooks/data/raw/match_scores.json


In [ ]:
# ── Load match scores ──────────────────────────────────────────────────────
with open(CFG['data']['match_scores_path']) as f:
    match_scores = json.load(f)

all_scores  = np.array([s for trk in match_scores.values() for s in trk.values()])
best_scores = np.array([max(trk.values()) for trk in match_scores.values()])

print(f"  Total track IDs  : {len(match_scores):,}")
print(f"  Total MIDI files : {len(all_scores):,}")

fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].hist(all_scores,  bins=80, alpha=0.7, label='all')
axes[0].hist(best_scores, bins=80, alpha=0.7, label='best/track')
axes[0].axvline(0.70, color='red', linestyle='--', label='thr=0.70')
axes[0].set_title('Match Score Distribution'); axes[0].legend()
axes[1].ecdf(best_scores); axes[1].axvline(0.70, color='red', linestyle='--')
axes[1].set_title('ECDF — best score per track')
plt.tight_layout(); plt.show()


FileNotFoundError: [Errno 2] No such file or directory: 'match_scores.json'

In [ ]:
# ── Initialise analyzer + linker ───────────────────────────────────────────
analyzer = MIDIHarmonicAnalyzer(
    key_window    = CFG['analyzer']['key_window'],
    key_confidence= CFG['analyzer']['key_confidence'],
)
linker = LakhMSDLinker(
    midi_root    = CFG['data']['midi_root'],
    h5_root      = CFG['data']['h5_root'],
    analyzer     = analyzer,
    match_scores = match_scores,
)
print('✓ Analyzer and linker ready.')

In [ ]:
# ── Build dataset (prototype: 50 tracks) ───────────────────────────────────
# Change max_tracks=None to process the full dataset (takes hours)
MAX_TRACKS = 50

df = linker.build_dataset(
    min_score   = CFG['dataset']['min_match_score'],
    pick_midi   = CFG['dataset']['pick_midi'],
    max_tracks  = MAX_TRACKS,
    with_harmonic_features = True,
    verbose     = True,
)

print(f'\nDataFrame shape : {df.shape}')
df.head(3)

In [ ]:
# ── Explore: genre distribution ────────────────────────────────────────────
genre_counts = df['primary_genre'].value_counts()
top_n = min(20, len(genre_counts))

genre_counts[:top_n].plot(kind='barh', figsize=(9, 5), title=f'Top {top_n} genres')
plt.xlabel('# songs'); plt.tight_layout(); plt.show()
print(genre_counts[:top_n])

In [ ]:
# ── Explore: key & mode distribution ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
df['global_key'].value_counts()[:12].plot(kind='bar', ax=axes[0], title='Detected key')
df['global_mode'].value_counts().plot(kind='pie', ax=axes[1], title='Mode', autopct='%1.0f%%')
plt.tight_layout(); plt.show()

In [ ]:
# ── Explore: harmonic feature distributions ────────────────────────────────
harm_cols = ['transition_entropy', 'chord_vocab_roman', 'num_modulations',
             'harm_rhythm_mean', 'func_ratio_T', 'func_ratio_D']
avail = [c for c in harm_cols if c in df.columns]

df[avail].hist(bins=20, figsize=(12, 6), layout=(2, 3))
plt.suptitle('Harmonic Feature Distributions', y=1.02)
plt.tight_layout(); plt.show()

In [ ]:
# ── Save dataset ───────────────────────────────────────────────────────────
import pathlib
out_dir = pathlib.Path(CFG['dataset']['output_dir'])
out_dir.mkdir(parents=True, exist_ok=True)

parquet_path = CFG['dataset']['parquet_file']
csv_path     = CFG['dataset']['csv_file']

df.to_parquet(parquet_path, index=False)
df.to_csv(csv_path, index=False)

show_paths([("Parquet", parquet_path), ("CSV", csv_path)])
